# 04 — Batch score

Loads the Unity Catalog model `ultimate_claim_severity` and scores all rows in
`claim_severity_features`, writing `{catalog}.{schema}.claim_severity_predictions`.

**Requires Dedicated ML Runtime** (`ml_cluster_id`).

### Step 1 — Configure scoring targets

Set the feature table, predictions output table, and model URIs. Prefer the Unity Catalog `Champion` alias; fall back to version `1` if the alias is not set yet.

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "ml")
dbutils.widgets.text("project_src", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
project_src = dbutils.widgets.get("project_src").rstrip("/")

feature_table = f"{catalog}.{schema}.claim_severity_features"
predictions_table = f"{catalog}.{schema}.claim_severity_predictions"
model_uri = f"models:/{catalog}.{schema}.ultimate_claim_severity@Champion"
model_uri_fallback = f"models:/{catalog}.{schema}.ultimate_claim_severity/1"
print(f"feature_table={feature_table}")
print(f"predictions_table={predictions_table}")

### Step 2 — Load the registered model

Point MLflow at the Databricks Unity Catalog registry, import the shared feature column list, and load the model as a pyfunc for batch inference.

In [ ]:
import sys
from pathlib import Path

import mlflow
import pandas as pd

candidates = [Path(project_src)]
if project_src and not project_src.startswith("/Workspace"):
    candidates.append(Path("/Workspace") / project_src.lstrip("/"))
for p in candidates:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
        break
else:
    raise FileNotFoundError(f"project_src not found: {project_src}")

from ml_pipeline_demo.features import MODEL_FEATURE_COLUMNS

mlflow.set_registry_uri("databricks-uc")

try:
    model = mlflow.pyfunc.load_model(model_uri)
    print(f"Loaded {model_uri}")
except Exception:
    model = mlflow.pyfunc.load_model(model_uri_fallback)
    print(f"Loaded {model_uri_fallback} (no Champion alias yet)")

### Step 3 — Ensure the Champion alias

For demo convenience, point the `Champion` alias at the latest registered model version when it is missing. Production jobs would typically manage aliases through a promotion workflow instead.

In [ ]:
from mlflow.tracking import MlflowClient

# Set Champion alias on the latest version when missing (demo convenience).
client = MlflowClient()
versions = client.search_model_versions(f"name='{catalog}.{schema}.ultimate_claim_severity'")
if not versions:
    raise RuntimeError("No registered model versions found. Run 03_train_and_evaluate first.")

latest = max(versions, key=lambda v: int(v.version))
try:
    client.set_registered_model_alias(
        name=f"{catalog}.{schema}.ultimate_claim_severity",
        alias="Champion",
        version=latest.version,
    )
    print(f"Set Champion -> version {latest.version}")
except Exception as exc:  # noqa: BLE001
    print(f"Alias note: {exc}")

### Step 4 — Score rows and write predictions

Score every row in the feature table, add `predicted_ultimate`, uplift vs first incurred, and absolute error, then overwrite `claim_severity_predictions`. Display the highest-uplift rows as triage candidates for claims review.

In [ ]:
# Score all feature rows and attach triage metrics (uplift vs first, abs error).
pdf = spark.table(feature_table).toPandas()
preds = model.predict(pdf[MODEL_FEATURE_COLUMNS])

out = pdf[
    [
        "claim_id",
        "policy_id",
        "first_incurred",
        "ultimate_incurred",
        "peril_type",
        "wind_risk_band",
        "building_type",
        "region_name",
    ]
].copy()
out["predicted_ultimate"] = preds
out["uplift_vs_first"] = out["predicted_ultimate"] / out["first_incurred"].clip(lower=1.0)
out["abs_error"] = (out["predicted_ultimate"] - out["ultimate_incurred"]).abs()

spark.createDataFrame(out).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(predictions_table)

print(f"Wrote {len(out)} rows to {predictions_table}")
print("Highest predicted uplift vs first incurred (triage candidates):")
display(out.sort_values("uplift_vs_first", ascending=False).head(10))
print("Batch score complete.")